In [38]:
# ===========================================================
# F1M1 — Módulo 1: EDA Leonali
# Notebook 08 — Análisis temporal: estacionalidad y tendencia
# ===========================================================

import pandas as pd
import plotly.express as px

df = pd.read_parquet('../data/processed/Dataset1_anonimizado.parquet')
print(df.shape)
print(df['FechaProceso'].min(), '→', df['FechaProceso'].max())

(581306, 15)
2012-12-31 00:00:00 → 2022-11-17 00:00:00


In [39]:
print('Fechas únicas:', df['FechaProceso'].nunique())
print('Rango días:', (df['FechaProceso'].max() - df['FechaProceso'].min()).days)
print('\nMuestra de fechas:')
print(df['FechaProceso'].sort_values().unique()[:20])

Fechas únicas: 3608
Rango días: 3608

Muestra de fechas:
<DatetimeArray>
['2012-12-31 00:00:00', '2013-01-01 00:00:00', '2013-01-02 00:00:00',
 '2013-01-03 00:00:00', '2013-01-04 00:00:00', '2013-01-05 00:00:00',
 '2013-01-06 00:00:00', '2013-01-07 00:00:00', '2013-01-08 00:00:00',
 '2013-01-09 00:00:00', '2013-01-10 00:00:00', '2013-01-11 00:00:00',
 '2013-01-12 00:00:00', '2013-01-13 00:00:00', '2013-01-14 00:00:00',
 '2013-01-15 00:00:00', '2013-01-16 00:00:00', '2013-01-17 00:00:00',
 '2013-01-18 00:00:00', '2013-01-19 00:00:00']
Length: 20, dtype: datetime64[us]


In [40]:
df['año'] = df['FechaProceso'].dt.year
df['mes'] = df['FechaProceso'].dt.month
df['trimestre'] = df['FechaProceso'].dt.quarter
df['dia_semana'] = df['FechaProceso'].dt.day_name()


In [41]:
fig = px.bar(
    df.groupby('mes')['Costo_Venta'].sum().reset_index(),
    x='mes',
    y='Costo_Venta',
    title='Venta por mes - Leonali'
)
fig.show()
fig.write_image('../reports/08_venta_por_mes.png', width=1200, height=500)

In [42]:
# Paso 1: venta por año-mes (ya lo tienes)
ventas_mes = df.groupby(['año', 'mes'])['Costo_Venta'].sum().reset_index()

# Paso 2: promedio anual por año
promedio_anual = df.groupby('año')['Costo_Venta'].sum().reset_index()
promedio_anual['promedio_mes'] = promedio_anual['Costo_Venta'] / 12

# Paso 3: unir y calcular índice
ventas_mes = ventas_mes.merge(promedio_anual[['año', 'promedio_mes']], on='año')
ventas_mes['indice'] = ventas_mes['Costo_Venta'] / ventas_mes['promedio_mes']

In [47]:
ventas_filtrado = ventas_mes[(ventas_mes['año'] >= 2013) & (ventas_mes['año'] <= 2021)]

fig = px.line(
    ventas_filtrado,
    x='mes',
    y='indice',
    color='año',
    title='Índice de Estacionalidad por Año - Leonali'
)
fig.show()
fig.write_image('../reports/08_estacionalidad_promedio.png', width=1200, height=500)

In [44]:
indice_promedio = ventas_filtrado.groupby('mes')['indice'].mean().reset_index()

fig = px.bar(
    indice_promedio,
    x='mes',
    y='indice',
    title='Índice de Estacionalidad Promedio - Leonali (2013-2021)'
)
fig.update_layout(yaxis_range=[0.7, 1.2])
fig.show()
fig.write_image('../reports/08_estacionalidad_poraño.png', width=1200, height=500)

In [45]:
df.groupby('dia_semana')['Costo_Venta'].sum().reset_index()

,dia_semana,Costo_Venta
0,Friday,4.229289e+08
1,Monday,5.210902e+08
2,Saturday,4.621254e+08
3,Sunday,4.179152e+08
4,Thursday,4.383675e+08
5,Tuesday,3.662841e+08
6,Wednesday,5.369162e+08


In [46]:
orden_dias = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
datos_dia = df.groupby('dia_semana')['Costo_Venta'].sum().reset_index()
fig = px.bar(
    datos_dia,
    x='dia_semana',
    y='Costo_Venta',
    title='Venta por día Semana - Leonali',
    category_orders={'dia_semana': orden_dias}
)
fig.show()
fig.write_image('../reports/08_venta_dia_semana.png', width=1200, height=500)